<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Data Carpentry · Python profesional</b></p>
      <p style="margin: 0;">Python profesional para analítica avanzada</p>
  </div>

</div>

# **Clean code, estructura de proyecto, nombres e imports**

Este laboratorio trabaja sobre una idea central: **que el código funcione no implica que esté bien diseñado**.

Seguiremos siempre el patrón:

> **ver el problema → diagnosticarlo → refactorizarlo → validar la mejora**

## Objetivos de aprendizaje

Al finalizar deberías poder reconocer y corregir nombres ambiguos, números mágicos, duplicación, funciones con demasiadas responsabilidades, estructuras de proyecto pobres, imports problemáticos y dependencias circulares. También crearás físicamente un pequeño paquete Python desde el propio notebook.

## Cómo trabajar

**Alumnado:** ejecuta la parte guiada y completa los doce ejercicios en orden. Sustituye los `TODO` y `pass` por tu código. Algunos ejercicios utilizan el paquete creado en los anteriores. Ejecutar las plantillas sin completarlas no demuestra que la práctica esté resuelta.

**Entorno:** Python, NumPy, Pandas y Matplotlib. Trabaja desde la carpeta de la sesión. Se crean áreas nuevas en `demos_clean_code/`; las soluciones añaden `practicas_clean_code/` y CSV de demostración. Los errores de imports y circularidad son deliberados y se capturan. Selecciona con el docente las actividades de aula; el resto puede servir de ampliación.


## 1. Preparación del entorno

In [ ]:
from pathlib import Path
import sys
import importlib
import shutil
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
SEED=42
rng=np.random.default_rng(SEED)
print("Entorno preparado")

## 2. Problema 1 — Nombres ambiguos

Primero veremos una versión que funciona pero obliga al lector a reconstruir el contexto.

In [ ]:
x = pd.DataFrame({
    "a": rng.integers(1, 8, 20),
    "b": rng.uniform(5, 100, 20),
    "c": rng.choice(["A", "B"], 20)
})
y = x["a"] * x["b"]
z = y.mean()
x.head(), z

### Diagnóstico

¿Qué significan `x`, `a`, `b`, `y` y `z`? El código es correcto sintácticamente pero su carga cognitiva es alta. La intención no está en el código.

## 2.1 Refactorización de nombres

In [ ]:
ventas = x.rename(columns={"a": "unidades", "b": "precio_unitario", "c": "canal"}).copy()
importe_venta = ventas["unidades"] * ventas["precio_unitario"]
importe_medio = importe_venta.mean()
np.testing.assert_allclose(importe_venta, y)
np.testing.assert_allclose(importe_medio, z)
ventas.head(), importe_medio


### Interpretación

La lógica es la misma, pero el lector ya no necesita recordar qué representa cada objeto. Un buen nombre reduce contexto externo.

## 3. Problema 2 — Números mágicos

In [ ]:
ventas_demo = pd.DataFrame({"importe": rng.gamma(2, 150, 50)})
ventas_demo["tipo"] = np.where(ventas_demo["importe"] > 500, "alta", "normal")
ventas_demo.head()

### Diagnóstico

`500` puede ser una regla de negocio, un valor provisional o una decisión normativa. El código no lo explica.

In [ ]:
UMBRAL_VENTA_ALTA = 500
ventas_demo["tipo"] = np.where(
    ventas_demo["importe"] > UMBRAL_VENTA_ALTA,
    "alta",
    "normal"
)
ventas_demo.head()

### Validación de la mejora

El valor sigue siendo el mismo, pero ahora la intención es explícita y el cambio está centralizado.

## 4. Problema 3 — Duplicación

In [ ]:
clientes_a = pd.DataFrame({"ingresos": [32000, 42000, 28000]})
clientes_b = pd.DataFrame({"ingresos": [51000, 26000, 39000]})
clientes_a["ingresos_mensuales"] = clientes_a["ingresos"] / 12
clientes_a["ingresos_altos"] = clientes_a["ingresos"] > 40000
clientes_b["ingresos_mensuales"] = clientes_b["ingresos"] / 12
clientes_b["ingresos_altos"] = clientes_b["ingresos"] > 40000
clientes_a, clientes_b

### Diagnóstico

La misma regla existe en dos lugares. Si mañana cambia el umbral, una copia podría actualizarse y la otra no.

In [ ]:
def enriquecer_clientes(datos: pd.DataFrame, umbral_ingresos_altos: float=40000) -> pd.DataFrame:
    salida=datos.copy()
    salida["ingresos_mensuales"] = salida["ingresos"] / 12
    salida["ingresos_altos"] = salida["ingresos"] > umbral_ingresos_altos
    return salida
clientes_a_ref=enriquecer_clientes(clientes_a[["ingresos"]])
clientes_b_ref=enriquecer_clientes(clientes_b[["ingresos"]])
clientes_a_ref, clientes_b_ref

### Interpretación

La transformación tiene ahora una única fuente de verdad. Esto aplica DRY cuando existe una misma intención conceptual.

## 5. Problema 4 — Función monolítica

In [ ]:
def proceso_monolitico(datos: pd.DataFrame) -> pd.DataFrame:
    salida=datos.copy()
    salida=salida.drop_duplicates()
    salida["importe"]=salida["unidades"]*salida["precio"]
    salida["venta_alta"]=salida["importe"]>500
    resumen=(salida.groupby("canal").agg(
        operaciones=("importe","size"),
        importe_medio=("importe","mean")
    ).reset_index())
    print(resumen)
    resumen.to_csv("resumen_temporal.csv", index=False)
    return resumen

### Diagnóstico

La función limpia, transforma, clasifica, agrega, imprime y exporta. Tiene demasiadas responsabilidades y por ello es difícil de probar o reutilizar parcialmente.

## 5.1 Separar responsabilidades

In [ ]:
def eliminar_duplicados(datos):
    return datos.drop_duplicates().copy()

def calcular_importe(datos):
    salida=datos.copy(); salida["importe"]=salida["unidades"]*salida["precio"]; return salida

def clasificar_ventas(datos, umbral=500):
    salida=datos.copy(); salida["venta_alta"]=salida["importe"]>umbral; return salida

def resumir_por_canal(datos):
    return datos.groupby("canal").agg(
        operaciones=("importe","size"),
        importe_medio=("importe","mean")
    ).reset_index()

## 5.2 Ejecutar y validar la versión modular

In [ ]:
ventas_base = pd.DataFrame({
    "unidades": rng.integers(1,8,200),
    "precio": rng.uniform(10,200,200),
    "canal": rng.choice(["web","tienda","partner"],200)
})
ventas_mod=eliminar_duplicados(ventas_base)
ventas_mod=calcular_importe(ventas_mod)
ventas_mod=clasificar_ventas(ventas_mod)
resumen_modular=resumir_por_canal(ventas_mod)
resumen_modular

### Interpretación

Cada función tiene ahora una responsabilidad principal. Eso mejora cohesión y reduce el acoplamiento entre tareas.

# 6. Crear automáticamente la estructura real del proyecto

In [ ]:
import tempfile
base_demos = Path("demos_clean_code")
base_demos.mkdir(exist_ok=True)
raiz = Path(tempfile.mkdtemp(prefix="proyecto_", dir=base_demos)).resolve()
carpetas = [raiz / "data/raw", raiz / "data/processed", raiz / "notebooks", raiz / "src/ventas_project", raiz / "tests"]
for carpeta in carpetas:
    carpeta.mkdir(parents=True, exist_ok=True)
print("Estructura creada sin borrar proyectos previos:", raiz)


## 6.1 Crear ficheros mínimos

In [ ]:
ficheros={
    raiz/"src"/"ventas_project"/"__init__.py":"",
    raiz/"README.md":"# Proyecto Clean Code\n",
    raiz/".gitignore":".venv/\n__pycache__/\n.ipynb_checkpoints/\n",
    raiz/"tests"/"__init__.py":""
}
for ruta,contenido in ficheros.items():
    ruta.write_text(contenido,encoding="utf-8")
print("Ficheros creados")

## 6.2 Inspeccionar lo creado

In [ ]:
for ruta in sorted(raiz.rglob("*")):
    nivel=len(ruta.relative_to(raiz).parts)-1
    sangria="    "*nivel
    simbolo="[DIR]" if ruta.is_dir() else "[FILE]"
    print(f"{sangria}{simbolo} {ruta.name}")

## 6.3 Validar programáticamente la estructura

In [ ]:
estado={str(carpeta):carpeta.exists() for carpeta in carpetas}
pd.Series(estado, name="existe")

### Interpretación

No damos por supuesto que la estructura existe: la verificamos. Esto convierte una instrucción teórica en un procedimiento reproducible.

# 7. Crear módulos Python reales desde el notebook

In [ ]:
validation_code = """
import pandas as pd

def validar_columnas(datos: pd.DataFrame, requeridas: list[str]) -> None:
    faltantes=[c for c in requeridas if c not in datos.columns]
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas: {faltantes}")
"""
features_code = """
import pandas as pd

def calcular_importe(datos: pd.DataFrame) -> pd.DataFrame:
    salida=datos.copy()
    salida["importe"]=salida["unidades"]*salida["precio"]
    return salida
"""
analysis_code = """
import pandas as pd

def resumir_por_canal(datos: pd.DataFrame) -> pd.DataFrame:
    return datos.groupby("canal").agg(
        operaciones=("importe","size"),
        importe_medio=("importe","mean")
    ).reset_index()
"""

In [ ]:
src_pkg=raiz/"src"/"ventas_project"
(src_pkg/"validation.py").write_text(validation_code,encoding="utf-8")
(src_pkg/"features.py").write_text(features_code,encoding="utf-8")
(src_pkg/"analysis.py").write_text(analysis_code,encoding="utf-8")
for archivo in sorted(src_pkg.glob("*.py")):
    print("-",archivo.name)

## 7.1 Verificar el contenido de un módulo

In [ ]:
print((src_pkg/"features.py").read_text(encoding="utf-8"))

### Interpretación

Las funciones ya existen fuera del notebook. Pueden versionarse, importarse y probarse de forma independiente.

# 8. Imports — primero veremos el fallo

In [ ]:
try:
    import ventas_project
except ModuleNotFoundError as error:
    print("Import fallido:")
    print(error)

### Diagnóstico

El paquete existe en disco, pero Python todavía no conoce `src/`. Existencia física e importabilidad son conceptos distintos.

## 8.1 Corregir temporalmente el path

In [ ]:
ruta_src=str((raiz/"src").resolve())
if ruta_src not in sys.path:
    sys.path.insert(0,ruta_src)
print(ruta_src)

## 8.2 Importar las funciones propias

In [ ]:
from ventas_project.validation import validar_columnas
from ventas_project.features import calcular_importe as calcular_importe_modulo
from ventas_project.analysis import resumir_por_canal as resumir_por_canal_modulo
print("Imports correctos")

## 8.3 Ejecutarlas

In [ ]:
datos_demo=pd.DataFrame({
    "unidades":[2,4,1],
    "precio":[10.0,7.5,30.0],
    "canal":["web","tienda","web"]
})
validar_columnas(datos_demo,["unidades","precio","canal"])
datos_demo=calcular_importe_modulo(datos_demo)
resumir_por_canal_modulo(datos_demo)

### Interpretación

El notebook ya se comporta como consumidor de código reusable: importa, ejecuta, visualiza e interpreta.

# 9. Imports problemáticos: `import *`

In [ ]:
print("from pandas import *")

### Diagnóstico

`import *` dificulta saber qué nombres entran en el namespace y aumenta colisiones. Preferimos imports explícitos.

## 9.1 Colisión de nombres — verla antes de corregirla

In [ ]:
from math import sqrt
sqrt_original=sqrt(16)
def sqrt(x):
    return "nombre sobrescrito"
sqrt_sobrescrito=sqrt(16)
sqrt_original, sqrt_sobrescrito

## 9.2 Corregir la colisión

In [ ]:
import math
math.sqrt(16)

### Interpretación

`math.sqrt` hace explícito el origen y reduce ambigüedad.

# 10. Imports absolutos y relativos

In [ ]:
print("Absoluto: from ventas_project.features import calcular_importe")
print("Relativo: from .features import calcular_importe")

### Interpretación

Los imports absolutos son muy explícitos; los relativos pueden ser útiles dentro de un paquete. Lo importante es mantener consistencia y claridad.

# 11. `__init__.py` y API pública

In [ ]:
init_code="""
from .features import calcular_importe
from .analysis import resumir_por_canal
"""
(src_pkg/"__init__.py").write_text(init_code,encoding="utf-8")
print((src_pkg/"__init__.py").read_text(encoding="utf-8"))

In [ ]:
import ventas_project
importlib.reload(ventas_project)
from ventas_project import calcular_importe, resumir_por_canal
print("API pública cargada")

### Interpretación

`__init__.py` puede exponer una interfaz cómoda y ocultar detalles internos del paquete.

# 12. Dependencia circular — provocar el problema

In [ ]:
circular_dir=raiz/"src"/"circular_demo"
circular_dir.mkdir(exist_ok=True)
(circular_dir/"__init__.py").write_text("",encoding="utf-8")
(circular_dir/"a.py").write_text("""
from .b import funcion_b

def funcion_a():
    return "A"
""",encoding="utf-8")
(circular_dir/"b.py").write_text("""
from .a import funcion_a

def funcion_b():
    return "B"
""",encoding="utf-8")
print("Dependencia circular creada")

## 12.1 Intentar importar

In [ ]:
try:
    import circular_demo.a
except Exception as error:
    print(type(error).__name__)
    print(error)

### Diagnóstico

Una dependencia circular suele señalar acoplamiento excesivo o una mala división de responsabilidades.

## 12.2 Refactorizar la lógica común

In [ ]:
(circular_dir/"common.py").write_text("""
def mensaje_base():
    return "Base común"
""",encoding="utf-8")
(circular_dir/"a.py").write_text("""
from .common import mensaje_base

def funcion_a():
    return f"A usa: {mensaje_base()}"
""",encoding="utf-8")
(circular_dir/"b.py").write_text("""
from .common import mensaje_base

def funcion_b():
    return f"B usa: {mensaje_base()}"
""",encoding="utf-8")
print("Circularidad eliminada")

## 12.3 Validar la corrección

In [ ]:
for nombre in ["circular_demo.a","circular_demo.b"]:
    sys.modules.pop(nombre,None)
from circular_demo.a import funcion_a
from circular_demo.b import funcion_b
print(funcion_a())
print(funcion_b())

### Interpretación

Extraer la dependencia común reduce acoplamiento y mejora cohesión.

# 13. Mini-práctica guiada — De script monolítico a paquete

In [ ]:
script_malo="""
import pandas as pd

df = pd.read_csv("ventas.csv")
df = df.drop_duplicates()
df["importe"] = df["unidades"] * df["precio"]
df["tipo"] = "normal"
df.loc[df["importe"] > 500, "tipo"] = "alta"
res = df.groupby("canal")["importe"].mean()
print(res)
res.to_csv("resumen.csv")
"""
print(script_malo)

### Diagnóstico

Ruta hardcodeada, ejecución global, nombres ambiguos, número mágico y mezcla de carga, limpieza, features, análisis y exportación.

## 13.1 Diseñar arquitectura

In [ ]:
print("""src/ventas_project/
├── io.py
├── cleaning.py
├── features.py
├── analysis.py
└── pipeline.py""")

## 13.2 Generar módulos automáticamente

In [ ]:
modulos_refactor={
"io.py":"""import pandas as pd

def cargar_csv(ruta):
    return pd.read_csv(ruta)
""",
"cleaning.py":"""def limpiar_duplicados(datos):
    return datos.drop_duplicates().copy()
""",
"features.py":"""def crear_importe(datos):
    salida=datos.copy()
    salida["importe"]=salida["unidades"]*salida["precio"]
    return salida
""",
"analysis.py":"""def resumen_canal(datos):
    return datos.groupby("canal").agg(operaciones=("importe","size"), importe_medio=("importe","mean")).reset_index()
""",
"pipeline.py":"""from .cleaning import limpiar_duplicados
from .features import crear_importe
from .analysis import resumen_canal

def ejecutar_pipeline(datos):
    datos=limpiar_duplicados(datos)
    datos=crear_importe(datos)
    return resumen_canal(datos)
"""
}
for nombre,contenido in modulos_refactor.items():
    (src_pkg/nombre).write_text(contenido,encoding="utf-8")
print("Módulos generados")

## 13.3 Inspeccionar los módulos

In [ ]:
for archivo in sorted(src_pkg.glob("*.py")):
    print("\n---",archivo.name,"---")
    print(archivo.read_text(encoding="utf-8"))

# 14. Visualizar la arquitectura conceptual antes y después

In [ ]:
categorias=["Carga","Limpieza","Features","Análisis","Exportación"]
monolitico=[1,1,1,1,1]
modular=[0,0,0,0,0]
fig,ax=plt.subplots(figsize=(8,4))
x=np.arange(len(categorias)); ancho=.35
ax.bar(x-ancho/2,monolitico,ancho,label="Script monolítico")
ax.bar(x+ancho/2,modular,ancho,label="Pipeline orquestador")
ax.set_xticks(x); ax.set_xticklabels(categorias,rotation=30)
ax.set_ylabel("Responsabilidad implementada directamente")
ax.set_title("Concentración de responsabilidades")
ax.legend(); plt.show()

### Interpretación del gráfico

La visualización es conceptual: el script monolítico implementa directamente todas las responsabilidades. En la versión modular, el orquestador delega cada tarea a un módulo especializado.

# 15. Checklist de clean code

- [ ] Los nombres comunican intención.
- [ ] No hay números mágicos sin justificar.
- [ ] No hay duplicación conceptual innecesaria.
- [ ] Cada función tiene una responsabilidad clara.
- [ ] La estructura separa datos, notebooks, código y tests.
- [ ] La lógica reusable vive en `src/`.
- [ ] Los imports son explícitos.
- [ ] No hay colisiones de nombres.
- [ ] No hay dependencias circulares.
- [ ] La API pública del paquete es clara.

# 16. Ejercicios finales


## Ejercicio 1 — Nombres

Refactoriza los nombres de `datos_ambiguos`, sus columnas `a`, `b`, `c` y los resultados `y_original` y `z_original` sin regenerar los datos. Comprueba que renombrar mantiene el importe y la media.


In [ ]:
datos_ambiguos = pd.DataFrame({'a': [2, 3, 1], 'b': [10.0, 20.0, 15.0], 'c': ['web', 'tienda', 'web']})
y_original = datos_ambiguos['a'] * datos_ambiguos['b']
z_original = y_original.mean()
# TODO: renombra las columnas y los resultados; comprueba equivalencia.


### Tu respuesta / interpretación

_Escribe aquí tu diagnóstico, decisiones y comprobaciones._


## Ejercicio 2 — Números mágicos

Clasifica 12 meses/año, un umbral de venta alta de 500 y la ruta de exportación como constante, parámetro o configuración. Implementa una clasificación parametrizable y comprueba los casos 499, 500 y 501.


In [ ]:
# TODO: declara MESES_POR_ANIO y una configuración de ruta.
def clasificar_importes(importes, umbral=500):
    # TODO: devuelve alta si importe > umbral; normal en otro caso.
    pass


### Tu respuesta / interpretación

_Escribe aquí tu diagnóstico, decisiones y comprobaciones._


## Ejercicio 3 — Duplicación

Muestra dos bloques de enriquecimiento de clientes y sustitúyelos por una función. Calcula ingresos mensuales y una marca de ingresos altos (> 40.000). Verifica equivalencia y ausencia de mutación.


In [ ]:
grupo_a = pd.DataFrame({"ingresos": [32000, 42000, 28000]})
grupo_b = pd.DataFrame({"ingresos": [51000, 26000, 39000]})
# TODO: escribe los dos bloques duplicados y refactorízalos.
def enriquecer_practica(datos, umbral=40000):
    pass


### Tu respuesta / interpretación

_Escribe aquí tu diagnóstico, decisiones y comprobaciones._


## Ejercicio 4 — Responsabilidad única

Separa limpieza, cálculo de importe, resumen y exportación. Compara con la función monolítica de la parte guiada y comprueba que el CSV contiene el mismo resumen.


In [ ]:
# TODO: define limpiar_practica, preparar_practica, resumir_practica y exportar_practica.
# TODO: compón las etapas y compara contra proceso_monolitico.


### Tu respuesta / interpretación

_Escribe aquí tu diagnóstico, decisiones y comprobaciones._


## Ejercicio 5 — Crear estructura automáticamente

Crea data/raw, data/processed, notebooks, src, tests y README.md con pathlib. Usa una carpeta nueva dentro de `practicas_clean_code` y comprueba existencia e idempotencia sin borrar archivos anteriores.


In [ ]:
# TODO: crea una carpeta de trabajo nueva y declara sus subcarpetas.
# TODO: crea README.md y comprueba todo con assert.


### Tu respuesta / interpretación

_Escribe aquí tu diagnóstico, decisiones y comprobaciones._


## Ejercicio 6 — Crear módulos

Genera validation.py, features.py y analysis.py dentro del paquete del ejercicio 5. La validación exige columnas y valores positivos y finitos de unidades y precio. Comprueba sintaxis de los módulos.


In [ ]:
# TODO: escribe __init__.py y los tres módulos.
# TODO: compila sus fuentes antes de intentar importarlos.


### Tu respuesta / interpretación

_Escribe aquí tu diagnóstico, decisiones y comprobaciones._


## Ejercicio 7 — Imports

Provoca y captura un ModuleNotFoundError para el paquete del ejercicio anterior. Explica la diferencia entre existir en disco y estar en sys.path. Corrígelo y prueba tanto datos válidos como inválidos.


In [ ]:
# TODO: captura ModuleNotFoundError sin ocultar errores distintos.
# TODO: añade src al path para la demostración y realiza imports explícitos.
# TODO: comprueba validación, transformación y resumen.


### Tu respuesta / interpretación

_Escribe aquí tu diagnóstico, decisiones y comprobaciones._


## Ejercicio 8 — Colisiones

Crea dos funciones con el mismo nombre dentro de módulos distintos. Muestra que un import directo sobrescribe el nombre local y corrige usando imports de módulos o alias explícitos.


In [ ]:
# TODO: crea dos módulos con una función calcular.
# TODO: muestra la sobrescritura y corrígela con alias.


### Tu respuesta / interpretación

_Escribe aquí tu diagnóstico, decisiones y comprobaciones._


## Ejercicio 9 — Dependencia circular

Crea dos módulos que se importen mutuamente antes de definir sus funciones. Captura el ImportError esperado y elimina el ciclo extrayendo una dependencia común a un tercer módulo.


In [ ]:
# TODO: crea el paquete circulo_practica y dos módulos con import circular.
# TODO: captura ImportError y extrae la lógica a common.py.
# TODO: limpia la caché del paquete de práctica y valida la corrección.


### Tu respuesta / interpretación

_Escribe aquí tu diagnóstico, decisiones y comprobaciones._


## Ejercicio 10 — __init__.py

Expón validar_ventas, preparar_ventas y resumir_ventas como API del paquete. Comprueba que los imports públicos funcionan.


In [ ]:
# TODO: escribe imports relativos en __init__.py y declara __all__.
# TODO: recarga el paquete y compara sus resultados con los imports internos.


### Tu respuesta / interpretación

_Escribe aquí tu diagnóstico, decisiones y comprobaciones._


## Ejercicio 11 — Arquitectura

Diseña un proyecto que consuma una API, limpie datos, calcule indicadores, genere gráficos y tenga tests. Describe dependencias y cómo ejecutar sin acceder a la API en cada prueba.


In [ ]:
# TODO: escribe una tabla de módulos, responsabilidades y dependencias.


### Tu respuesta / interpretación

_Escribe aquí tu diagnóstico, decisiones y comprobaciones._


## Ejercicio 12 — Reto final

Revisa el script monolítico siguiente (más de 40 líneas). Detecta nombres, duplicación, números mágicos y responsabilidades mezcladas. Crea un paquete con validación, limpieza, features, análisis, exportación y pipeline. Conserva el resumen y los CSV originales.


In [ ]:
script_monolitico_reto = 'from pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nr = np.random.default_rng(42)\nd = pd.DataFrame({\n    "id": np.arange(1, 1201),\n    "u": r.integers(1, 8, 1200),\n    "p": r.uniform(5, 150, 1200),\n    "c": r.choice(["web", "tienda", "partner"], 1200),\n    "descuento": r.choice([0.0, 0.1, 0.2], 1200),\n})\nd = pd.concat([d, d.iloc[[0, 1]]], ignore_index=True)\nd = d.drop_duplicates()\nd["importe"] = d["u"] * d["p"]\nd["neto"] = d["importe"] * (1 - d["descuento"])\nd["alta"] = d["neto"] > 500\nw = d.loc[d["c"] == "web"].copy()\nt = d.loc[d["c"] == "tienda"].copy()\nw["importe"] = w["u"] * w["p"]\nw["neto"] = w["importe"] * (1 - w["descuento"])\nt["importe"] = t["u"] * t["p"]\nt["neto"] = t["importe"] * (1 - t["descuento"])\nx = d.groupby("c", as_index=False).agg(\n    operaciones=("id", "size"),\n    importe_neto=("neto", "sum"),\n    pct_alta=("alta", "mean"),\n)\nx = x.rename(columns={"c": "canal"})\nprint(x)\na = Path("salida_monolitica")\na.mkdir(exist_ok=True)\nx.to_csv(a / "resumen.csv", index=False)\nw.to_csv(a / "web.csv", index=False)\nt.to_csv(a / "tienda.csv", index=False)\nfig, ax = plt.subplots(figsize=(7, 4))\nax.bar(x["canal"], x["importe_neto"])\nax.set_title("Importe neto por canal")\nax.set_xlabel("Canal")\nax.set_ylabel("Importe neto")\nplt.show()\nfig, ax = plt.subplots(figsize=(7, 4))\nax.hist(d["neto"], bins=25)\nax.set_title("Distribución de importe neto")\nax.set_xlabel("Importe neto")\nax.set_ylabel("Operaciones")\nplt.show()\n'
print(script_monolitico_reto)
assert len(script_monolitico_reto.splitlines()) >= 40


In [ ]:
# TODO: analiza el script y ejecuta una referencia en un namespace separado.
# TODO: genera módulos e importa el pipeline.
# TODO: compara datos, resumen y exportaciones; comprueba que la entrada no cambia.


### Tu respuesta / interpretación

_Escribe aquí tu diagnóstico, decisiones y comprobaciones._


# 17. Preguntas de reflexión


1. ¿Puede un código funcionar y ser de mala calidad?
2. ¿Cuándo un nombre es demasiado corto?
3. ¿DRY significa eliminar cualquier repetición?
4. ¿Qué diferencia hay entre cohesión y acoplamiento?
5. ¿Por qué una función con muchas responsabilidades es difícil de testear?
6. ¿Qué aporta separar `src/` y `notebooks/`?
7. ¿Qué riesgo tiene `import *`?
8. ¿Qué suele indicar una dependencia circular?
9. ¿Crear más archivos garantiza mejor diseño?
10. ¿Qué parte de este notebook debería quedarse como notebook y qué parte debería trasladarse a módulos?

## Tus respuestas

1. …
2. …
3. …
4. …
5. …
6. …
7. …
8. …
9. …
10. …
